# Practica 5 - Sesion 3: Comparación de Clasificadores

## Ingeniería del Conocimiento    2025/2026
### Prof. Juan A. Recio García

En esta práctica compararemos distintos clasificadores de **scikit-learn** y **XGBoost** sobre tres datasets con contextos muy diferentes. El objetivo central es entender que **la métrica de evaluación debe elegirse según el coste real de equivocarse**, no siempre la accuracy.

## Clasificadores a comparar

| Clasificador | Clase sklearn |
|---|---|
| Logistic Regression | `LogisticRegression` |
| Naive Bayes | `GaussianNB` |
| K-Nearest Neighbors | `KNeighborsClassifier` |
| SVM | `SVC` |
| MLP (red neuronal) | `MLPClassifier` |
| Random Forest | `RandomForestClassifier` |
| XGBoost | `XGBClassifier` |


## Datasets y métricas

| # | Dataset | Dominio | Métrica principal | Justificación |
|---|---|---|---|---|
| 1 | **Credit Card Fraud** | Finanzas | **F1-score** | Clases muy desbalanceadas; accuracy sería engañosa |
| 2 | **Wisconsin Breast Cancer** | Medicina | **Recall** | Minimizar falsos negativos: no dejar cáncer sin detectar |
| 3 | **Stock Movement** | Bolsa | **Precision** | Solo operar cuando la predicción es muy fiable |

---

**El Dataset 1 (Credit Card Fraud) está resuelto como ejemplo.**  
Para los datasets 2 y 3 debéis replicar la misma estructura rellenando las celdas marcadas con `# TODO`.

---
## 0. Imports y configuración general

In [ ]:
# Posiblemente tendrás que instalar XGBoost para ejecutar este notebook ya que no se incluye en sklearn. Puedes hacerlo con:
!pip install xgboost jinja2

In [ ]:
# ── Librerías de análisis de datos ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Preprocesamiento ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

# ── Clasificadores ────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB           # Clasificador bayesiano probabilístico
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import jinja2
from imblearn.over_sampling import SMOTE

# ── Métricas de evaluación ────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# ── Configuración visual ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)
import warnings
warnings.filterwarnings('ignore')  # Suprimimos warnings de convergencia durante la práctica


### Funciones auxiliares reutilizables
Definimos aquí las funciones que usaremos en los tres datasets para evitar repetir código.

In [ ]:
def evaluar_clasificadores(X_train, X_test, y_train, y_test, highlight_metric='f1'):
    """
    Entrena los 7 clasificadores sobre los datos dados y devuelve una tabla con
    accuracy, precision, recall y F1. La columna highlight_metric aparece resaltada.
    """
    modelos = {
        'Logistic Regression' : LogisticRegression(max_iter=1000),
        'Naive Bayes'         : GaussianNB(),
        'KNN'                 : KNeighborsClassifier(n_neighbors=5),
        'SVM'                 : SVC(),
        'MLP'                 : MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, random_state=42),
        'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost'             : XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    }

    resultados = []
    for nombre, modelo in modelos.items():
        print(f'Entrenando {nombre}...')
        modelo.fit(X_train, y_train)       # Entrenamos el modelo con los datos de entrenamiento
        y_pred = modelo.predict(X_test)    # Generamos predicciones sobre el conjunto de test
        resultados.append({
            'Modelo'    : nombre,
            'Accuracy'  : accuracy_score(y_test, y_pred),
            'Precision' : precision_score(y_test, y_pred, zero_division=0),
            'Recall'    : recall_score(y_test, y_pred, zero_division=0),
            'F1'        : f1_score(y_test, y_pred, zero_division=0),
        })

    col_map = {'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1', 'accuracy': 'Accuracy'}
    col = col_map.get(highlight_metric, 'F1')
    df = pd.DataFrame(resultados).set_index('Modelo').sort_values(col, ascending=False)
    # Resaltamos en verde la columna relevante para que destaque visualmente
    return df.style.background_gradient(subset=[col], cmap='Greens').format('{:.4f}')


def mostrar_confusion(modelo, X_test, y_test, titulo='Matriz de Confusión'):
    """Muestra la matriz de confusión y el classification report de un modelo ya entrenado."""
    y_pred = modelo.predict(X_test)
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred)).plot(cmap='Blues')
    plt.title(titulo)
    plt.tight_layout()
    plt.show()
    print(classification_report(y_test, y_pred))
    print('Accuracy',  accuracy_score(y_test, y_pred))
    print('Precision', precision_score(y_test, y_pred, zero_division=0))
    print('Recall',    recall_score(y_test, y_pred, zero_division=0))
    print('F1',        f1_score(y_test, y_pred, zero_division=0))



---
# Dataset 1 — Credit Card Fraud *(Ejemplo resuelto)*
## Métrica principal: **F1-score**

> **¿Por qué F1?**  
> El dataset está muy desbalanceado (~0.17% de fraudes). La *accuracy* sería engañosa:  
> un modelo que prediga siempre "no fraude" tendría >99% de accuracy pero no detectaría ningún fraude real.  
> El **F1** es la media armónica de Precision y Recall, ideal cuando las clases están desbalanceadas:
>
> $$F1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Archivo:** `creditcard.csv`  
Fuente: [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

### 1.1 Carga y exploración

In [ ]:
# Las columnas V1-V28 son componentes PCA (anonimizadas por el banco)
# 'Amount' es el importe de la transacción; 'Class' es la etiqueta (1=fraude, 0=legítima)
df_fraud = pd.read_csv('creditcard.csv')

print(f'Shape: {df_fraud.shape}')
print(f'Valores nulos totales: {df_fraud.isnull().sum().sum()}')
df_fraud.head(3)

In [ ]:
# Analizamos el desbalance — esto justifica elegir F1 en vez de accuracy
conteo = df_fraud['Class'].value_counts()
print(f'Transacciones legítimas   : {conteo[0]:,}')
print(f'Transacciones fraudulentas: {conteo[1]:,}')
print(f'Porcentaje de fraude      : {100 * conteo[1] / len(df_fraud):.3f}%')

### 1.2 Preprocesamiento

In [ ]:
# Eliminamos 'Time' (no aporta información útil) y separamos features de la variable objetivo
X_fraud = df_fraud.drop(columns=['Time', 'Class']).copy()
y_fraud = df_fraud['Class']

# Escalamos los datos. A
# Aunque los clasificadores basados en árboles no lo requieren, otros como SVM o MLP sí se benefician de tener las features en la misma escala.
# StandardScaler centra en media=0 y desviación estándar=1
scaler_fraud = StandardScaler()
X_fraud = scaler_fraud.fit_transform(X_fraud)

# Split estratificado: preserva el mismo porcentaje de fraudes (~0.17%) en train y test
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

print(f'Train: {X_train_f.shape} | Test: {X_test_f.shape}')
print(f'Fraudes en train: {y_train_f.sum()} | Fraudes en test: {y_test_f.sum()}')

### 1.3 Entrenamiento y evaluación de clasificadores

In [ ]:
# Entrenamos los 7 clasificadores y mostramos la tabla comparativa
# La columna F1 aparece resaltada porque es la métrica relevante para este problema
# SVM y MLP pueden tardar 1-2 minutos con datasets grandes
tabla_fraud = evaluar_clasificadores(X_train_f, X_test_f, y_train_f, y_test_f, highlight_metric='f1')
tabla_fraud

### 1.4 Análisis del mejor modelo

In [ ]:
# Entrenamos XGBoost para analizar en detalle (suele destacar en datos tabulares desbalanceados)
# Si en tu tabla un modelo diferente ha obtenido mejor F1, sustitúyelo aquí
# XGBoost cuenta con un parámetro muy útil que es scale_pos_weight. Con ello podemos intentar mejorar el recall sin sacrificar demasiado precision. Normalmente hay que probar con distintos valores hasta encontrar el que mejor se adapta a tu dataset. En este caso, con un valor de 0.50 hemos conseguido mejorar el recall sin sacrificar demasiado precision, lo que se traduce en una mejora del F1. 
mejor_f = XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0, scale_pos_weight=.50)
mejor_f.fit(X_train_f, y_train_f)

# La matriz de confusión muestra exactamente cuántos fraudes se detectan vs cuántos se pierden
mostrar_confusion(mejor_f, X_test_f, y_test_f,
                  titulo='Matriz de Confusión — XGBoost — Credit Card Fraud')

> Prueba con otros valores de scale_pos_weight
>
> ¿Qué otra alternativa podríamos haber usado para evitar el desbalanceo?

In [ ]:
# Eliminamos 'Time' (no aporta información útil) y separamos features de la variable objetivo
X_fraud = df_fraud.drop(columns=['Time', 'Class']).copy()
y_fraud = df_fraud['Class']

# Escalamos los datos. A
# Aunque los clasificadores basados en árboles no lo requieren, otros como SVM o MLP sí se benefician de tener las features en la misma escala.
# StandardScaler centra en media=0 y desviación estándar=1
scaler_fraud = StandardScaler()
X_fraud = scaler_fraud.fit_transform(X_fraud)

# Split estratificado: preserva el mismo porcentaje de fraudes (~0.17%) en train y test
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

smote = SMOTE(random_state=42)
X_train_f, y_train_f = smote.fit_resample(X_train_f, y_train_f)
y_train_f.value_counts()

In [ ]:
# Cuidado, ahora tarda un buen rato porque hemos aumentado el número de muestras de fraude en el conjunto de entrenamiento. 
# Si quieres acelerar el proceso, puedes probar a reducir el número de árboles en Random Forest o XGBoost, además del número de iteraciones en SVM o MLP, o a entrenar solo un subconjunto de los clasificadores.
# o a entrenar solo un subconjunto de los clasificadores.
tabla_fraud = evaluar_clasificadores(X_train_f, X_test_f, y_train_f, y_test_f, highlight_metric='f1')
tabla_fraud

### 1.5 Conclusiones — Credit Card Fraud
>
> *Escribe aquí tus conclusiones:*
>
>- ¿Qué modelo obtiene el mejor F1 para detectar fraudes?
>- ¿Cómo se comporta la accuracy frente al F1? ¿Son coherentes?
>- ¿Qué ocurriría si priorizáramos Recall sobre Precision en este contexto?
>- ¿Sirve de algo el oversampling? ¿Por qué?

---
# Dataset 2 — Wisconsin Breast Cancer  *(A completar)*
## Métrica principal: **Recall**

> **¿Por qué Recall?**  
> En diagnóstico médico, un **falso negativo** (no detectar un cáncer real) es mucho más grave  
> que un falso positivo (hacer más pruebas a alguien sano). El Recall mide la proporción  
> de enfermos reales que el modelo es capaz de identificar:
>
> $$\text{Recall} = \frac{TP}{TP + FN}$$

Fuente: [UCI ML Repository / sklearn.datasets.load_breast_cancer](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)

### 2.1 Carga y exploración

In [ ]:
# sklearn incluye un dataset de cáncer de mama que es muy utilizado para practicar clasificación binaria. 
# Es un dataset limpio, sin valores nulos, con 569 muestras y 30 features numéricas extraídas de imágenes de tumores. 
# La variable objetivo es 'target', donde 0 indica tumores benignos y 1 indica tumores malignos. 
# TODO: Carga el dataset y analiza su contenido
# Muestra: shape, y distribución de clases (maligno=1 vs benigno=0)

df_cancer = load_breast_cancer(as_frame=True).frame
df_cancer.head(3)
# ...

### 2.2 Preprocesamiento

In [ ]:
# TODO: Separa X e y, aplica StandardScaler y realiza el split estratificado
# Recuerda: usa stratify=y para mantener la proporción de clases en train y test

# X_cancer = ...
# y_cancer = ...
# scaler_cancer = StandardScaler()
# X_cancer_scaled = scaler_cancer.fit_transform(X_cancer)
# X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(...)

### 2.3 Entrenamiento y evaluación de clasificadores

In [ ]:
# TODO: Llama a evaluar_clasificadores indicando recall como métrica principal

# tabla_cancer = evaluar_clasificadores(X_train_c, X_test_c, y_train_c, y_test_c, highlight_metric='recall')
# tabla_cancer

### 2.4 Análisis del mejor modelo

In [ ]:
# TODO: Instancia y entrena el modelo que haya obtenido mayor Recall
# Muestra su matriz de confusión con mostrar_confusion()
# Intenta mejorarlo ajustando sus hiperparámetros o aplicando técnicas de preprocesamiento como SMOTE o ajuste de pesos de clase.

# mejor_c = ...
# mejor_c.fit(X_train_c, y_train_c)
# mostrar_confusion(mejor_c, X_test_c, y_test_c, titulo='...')

### 2.5 Conclusiones — Wisconsin Breast Cancer

*Escribe aquí tus conclusiones:*

- ¿Qué modelo maximiza el Recall?
- ¿Cuántos casos de cáncer quedan sin detectar (falsos negativos) con ese modelo?
- ¿Existe un trade-off entre Recall alto y Precision baja en algún modelo? ¿Es aceptable en este contexto?

---
# Dataset 3 — Stock Movement  *(A completar)*
## Métrica principal: **Precision**

> **¿Por qué Precision?**  
> En predicción bursátil queremos operar **solo cuando el modelo está muy seguro** de que el precio subirá.  
> Un **falso positivo** (invertir cuando el precio baja) tiene coste económico directo.  
> La Precision mide cuántas de nuestras señales de compra son realmente correctas:
>
> $$\text{Precision} = \frac{TP}{TP + FP}$$

📂 **Archivo:** `stock_movement.csv`  
📎 Target: `1` = el precio sube al día siguiente, `0` = baja o se mantiene.

### 3.1 Carga y exploración

In [ ]:
# TODO: Carga el dataset stock_movement.csv y analiza su contenido
# Muestra: shape, valores nulos, y distribución de clases (sube vs baja)

df_stock = pd.read_csv('stock_movement_small.csv')
display(df_stock.head(3))
print(f'Shape: {df_stock.shape}')
print(f'Distribución de clases:\n{df_stock["target"].value_counts()}')

### 3.2 Preprocesamiento

In [ ]:
# TODO: Separa X e y, aplica StandardScaler y realiza el split
#
# 💡 Reflexión importante: los datos de bolsa son una serie temporal.
# Un split aleatorio mezclaría datos del futuro con el pasado (data leakage).
# En series temporales se usa shuffle=False para respetar el orden cronológico.

# X_stock = ...
# y_stock = ...
# scaler_stock = StandardScaler()
# X_stock_scaled = scaler_stock.fit_transform(X_stock)
# X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(..., shuffle=False)

### 3.3 Entrenamiento y evaluación de clasificadores

In [ ]:
# TODO: Llama a evaluar_clasificadores indicando precision como métrica principal

# tabla_stock = evaluar_clasificadores(X_train_s, X_test_s, y_train_s, y_test_s, highlight_metric='precision')
# tabla_stock

### 3.4 Análisis del mejor modelo

In [ ]:
# TODO: Instancia y entrena el modelo que haya obtenido mayor Precision
# Muestra su matriz de confusión con mostrar_confusion()

# mejor_s = ...
# mejor_s.fit(X_train_s, y_train_s)
# mostrar_confusion(mejor_s, X_test_s, y_test_s, titulo='...')

### 3.5 Conclusiones — Stock Movement

*Escribe aquí tus conclusiones:*

>- ¿Qué modelo obtiene mayor Precision?
>- ¿Qué ocurre con el Recall de ese modelo? ¿Es un problema en este contexto?
>- ¿Podría el modelo estar sobreajustando? ¿Cómo lo detectarías?

---
# Comparativa Global

Completa la tabla resumen con los mejores resultados obtenidos en cada dataset.

In [ ]:
# TODO: Rellena los valores con los resultados de tu ejecución
resumen = pd.DataFrame([
    {'Dataset': 'Credit Card Fraud', 'Métrica clave': 'F1',        'Mejor Modelo': '???', 'Valor': 0.0},
    {'Dataset': 'Breast Cancer',     'Métrica clave': 'Recall',    'Mejor Modelo': '???', 'Valor': 0.0},
    {'Dataset': 'Stock Movement',    'Métrica clave': 'Precision', 'Mejor Modelo': '???', 'Valor': 0.0},
])
resumen.style.background_gradient(subset=['Valor'], cmap='Greens').format({'Valor': '{:.4f}'})

### Preguntas de reflexión final

*Responde brevemente en esta celda:*

**1.** ¿Existe un clasificador que gane en los tres datasets, o el mejor modelo depende siempre del problema?

**2.** ¿Por qué la accuracy no es una buena métrica universal? Da un ejemplo concreto con uno de los tres datasets.

**3.** ¿Qué clasificadores han sido más lentos de entrenar? ¿Merece la pena el coste computacional extra?

**4.** En el contexto médico, si tuvieras que elegir entre estos dos modelos hipotéticos, ¿cuál elegirías y por qué?

| Modelo | Recall | Precision |
|---|---|---|
| A | 0.99 | 0.60 |
| B | 0.85 | 0.95 |